In [1]:
import pandas as pd
import os

# Path to your data dir
data_dir = '/share/crsp/lab/pkaiser/ddlin/mids/datasci-266/mmiv/physionet.org/files/mimiciv/3.1/hosp'

# Function to load a gzipped CSV (with optional sampling for quick EDA)
def load_mimic_table(filename, nrows=None, sample_frac=0.1):
    filepath = os.path.join(data_dir, filename)
    if not os.path.exists(filepath):
        print(f"File {filepath} not found!")
        return None
    
    print(f"Loading {filename} (sampled: {sample_frac if sample_frac else 'full'})...")
    df = pd.read_csv(filepath, compression='gzip', nrows=nrows)  # Or sample: low_memory=False for big files
    
    if sample_frac:
        df = df.sample(frac=sample_frac, random_state=42)  # Random sample for speed
    
    print(f"Shape: {df.shape}")
    print(df.head())
    print(df.info())  # Quick dtypes/sizes
    return df

# Example: Load admissions (core table for hospitalizations)
admissions = load_mimic_table('admissions.csv.gz', sample_frac=0.05)  # 5% sample ~26K rows

# Example: Load patients (demographics)
patients = load_mimic_table('patients.csv.gz', sample_frac=0.05)

# Basic EDA: Merge on subject_id for cohort overview
if admissions is not None and patients is not None:
    cohort = admissions.merge(patients[['subject_id', 'gender', 'anchor_age']], on='subject_id', how='left')
    print("\nCohort summary:")
    print(cohort.groupby('gender')['anchor_age'].agg(['mean', 'count']))
    print(cohort['admission_type'].value_counts())

# For larger tables (e.g., labevents.csv.gz ~10M rows), use nrows=10000 for prototyping
# labs_sample = load_mimic_table('labevents.csv.gz', nrows=10000)

Loading admissions.csv.gz (sampled: 0.05)...
Shape: (27301, 16)
        subject_id   hadm_id            admittime            dischtime  \
464946    18521354  20755423  2121-05-08 02:56:00  2121-05-13 18:15:00   
358015    16559252  21482324  2174-03-12 21:57:00  2174-03-16 18:25:00   
395082    17237709  21276544  2157-11-04 20:35:00  2157-11-08 14:30:00   
513229    19397801  21268011  2165-03-29 04:45:00  2165-03-29 10:01:00   
297648    15463124  22494617  2181-04-24 00:31:00  2181-04-25 01:59:00   

       deathtime  admission_type admit_provider_id     admission_location  \
464946       NaN        EW EMER.            P595QV  WALK-IN/SELF REFERRAL   
358015       NaN        EW EMER.            P56LDX     PHYSICIAN REFERRAL   
395082       NaN        EW EMER.            P46834         EMERGENCY ROOM   
513229       NaN  EU OBSERVATION            P36VUP         EMERGENCY ROOM   
297648       NaN        EW EMER.            P42XKM         EMERGENCY ROOM   

       discharge_location in

In [3]:
os.getcwd()

'/share/crsp/lab/pkaiser/ddlin/mids/datasci-266/mmiv/notebooks'

In [4]:
import pandas as pd
import os
from datetime import timedelta

# Base directory (adjust if needed; based on your terminal path)
base_dir = '../'
hosp_dir = os.path.join(base_dir, 'physionet.org/files/mimiciv/3.1/hosp')
note_dir = os.path.join(base_dir, 'physionet.org/files/mimic-iv-note/2.2/note')
ext_dir = os.path.join(base_dir, 'physionet.org/files/mimic-iv-ext-22mcts/1.0.0')

# Key file paths for 30-day readmission task
files = {
    'admissions': os.path.join(hosp_dir, 'admissions.csv.gz'),
    'patients': os.path.join(hosp_dir, 'patients.csv.gz'),
    'diagnoses_icd': os.path.join(hosp_dir, 'diagnoses_icd.csv.gz'),
    'discharge': os.path.join(note_dir, 'discharge.csv.gz'),
    'events': os.path.join(ext_dir, 'clinical_event_timestamp.csv')  # Not gzipped
}

# Function to load, summarize, and check for readmission relevance
def load_and_check(filepath, nrows=1000, sample_frac=0.1, is_large=False):
    print(f"\n=== Loading {os.path.basename(filepath)} ===")
    if not os.path.exists(filepath):
        print(f"❌ File not found: {filepath}")
        return None
    
    try:
        kwargs = {'compression': 'gzip'} if filepath.endswith('.gz') else {}
        if is_large:
            df = pd.read_csv(filepath, nrows=nrows, **kwargs)
        else:
            df = pd.read_csv(filepath, **kwargs)
            if sample_frac and len(df) > 1000:
                df = df.sample(frac=sample_frac, random_state=42).reset_index(drop=True)
        
        print(f"✅ Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print("\nSample data:\n", df.head(3))
        print("\nData types:\n", df.dtypes)
        print("\nBasic stats:\n", df.describe(include='all').round(2))
        
        # Task-specific checks
        if 'admissions' in filepath:
            df['admittime'] = pd.to_datetime(df['admittime'])
            df['dischtime'] = pd.to_datetime(df['dischtime'])
            df = df.sort_values(['subject_id', 'admittime'])
            df['next_admittime'] = df.groupby('subject_id')['admittime'].shift(-1)
            df['days_to_next'] = (df['next_admittime'] - df['dischtime']).dt.days
            readmit_rate = (df['days_to_next'] <= 30).mean()
            print(f"\nReadmission preview (≤30 days): {readmit_rate:.2%} ({(df['days_to_next'] <= 30).sum()}/{len(df)})")
            print("Sample readmission gaps:\n", df['days_to_next'].dropna().head())
        elif 'patients' in filepath:
            print(f"\nAdult patients (anchor_age >=18): {(df['anchor_age'] >= 18).sum()}/{len(df)}")
        elif 'diagnoses_icd' in filepath:
            print(f"\nTop ICD codes: \n{df['icd_code'].value_counts().head()}")
        elif 'discharge' in filepath:
            print(f"\nAvg text length: {df['text'].str.len().mean():.0f} chars (sample: {df['text'].str[:100].iloc[0]}...)")
        elif 'events' in filepath:
            print(f"\nTop events: \n{df['Event'].value_counts().head()}")
            print(f"Timestamp range: {df['Time'].min()} to {df['Time'].max()} hours")
        
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Load key files for readmission task
admissions_df = load_and_check(files['admissions'], nrows=None)  # Full for label computation
patients_df = load_and_check(files['patients'])
diagnoses_df = load_and_check(files['diagnoses_icd'], nrows=5000)  # More rows for codes
discharge_df = load_and_check(files['discharge'], nrows=500)  # Text-heavy, small sample
events_df = load_and_check(files['events'], nrows=10000, is_large=True)  # 22M rows, heavy sample

# Quick merge preview for cohort (small samples)
print("\n=== Cohort Preview (Admissions + Patients + Discharge) ===")
if all(df is not None for df in [admissions_df, patients_df, discharge_df]):
    # Sample for merge
    adm_samp = admissions_df.sample(1000, random_state=42)
    pat_samp = patients_df[['subject_id', 'gender', 'anchor_age']]
    dis_samp = discharge_df[['hadm_id', 'text']].sample(100, random_state=42)  # Fewer texts
    cohort = adm_samp.merge(pat_samp, on='subject_id', how='left')
    cohort = cohort.merge(dis_samp, on='hadm_id', how='left')
    print(f"Shape after merge: {cohort.shape}")
    print(cohort[['subject_id', 'hadm_id', 'gender', 'anchor_age', 'admission_type', 'text']].head(2))
    print("\nText availability: ", (cohort['text'].notna()).mean())

# Save small cohort for further EDA if needed
# cohort.to_csv('readmission_cohort_sample.csv', index=False)
print("\n✅ EDA complete! Use 'cohort' for plots (e.g., cohort['anchor_age'].hist())")


=== Loading admissions.csv.gz ===
✅ Shape: (54603, 16)
Columns: ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']

Sample data:
    subject_id   hadm_id            admittime            dischtime deathtime  \
0    18521354  20755423  2121-05-08 02:56:00  2121-05-13 18:15:00       NaN   
1    16559252  21482324  2174-03-12 21:57:00  2174-03-16 18:25:00       NaN   
2    17237709  21276544  2157-11-04 20:35:00  2157-11-08 14:30:00       NaN   

  admission_type admit_provider_id     admission_location discharge_location  \
0       EW EMER.            P595QV  WALK-IN/SELF REFERRAL               HOME   
1       EW EMER.            P56LDX     PHYSICIAN REFERRAL   HOME HEALTH CARE   
2       EW EMER.            P46834         EMERGENCY ROOM               HOME   

  insurance      language marital